## Midterm

In [ ]:
import Exam as exam
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import t, norm, multivariate_normal, skew, kurtosis, spearmanr
import statsmodels.miscmodels.tmodel as tmodel
from statsmodels.stats.correlation_tools import cov_nearest, corr_nearest

#### 1. (20 pts)
Using problem1.csv: </br>
a. (2) Calculate the mean, standard deviation, skewness, and kurtosis of the data.

In [140]:
# load data
data1 = np.loadtxt('problem1.csv', skiprows=1)

In [141]:
mean = np.mean(data1)
std = np.std(data1, ddof=1)
skewness = skew(data1)
kurt = kurtosis(data1)
print(mean, std, skewness, kurt)

-0.0005316702326786276 0.0317278378920395 -0.2395102766006733 0.6804386235881297


b. (3) Given a choice between a normal distribution and a t-distribution, which one would you choose to
model the data? Why?

I would choose the T distribution because Kurtosis is larger then 0.

c. (4) Fit both distributions and prove or disprove your choice in part (b).

In [142]:
# normal 
est_mu = np.mean(data1)
est_sigma = np.std(data1, ddof=1)
print(est_mu, est_sigma)

-0.0005316702326786276 0.0317278378920395


In [143]:
# T
est_nu, est_mu, est_sigma = t.fit(data1)
print(est_nu, est_mu, est_sigma)

13.711803328219464 -0.00021915131437091594 0.029299141739475267


In [144]:
AIC_n, BIC_n, AICc_n, AIC_t, BIC_t, AICc_t = exam.model_selection_metrics(data1)
print(AICc_n, AICc_t)

-4060.2325305235076 -4068.1094802192783


T distribution is better with a lower AICc.

d. (4) Calculate the 5% and 1% VaR and ES for each distribution.

In [145]:
# VaR & ES from normal dist
def var_es_normal(data, alpha):
    est_mu = np.mean(data)
    est_sigma = np.std(data, ddof=1)
    
    z_score = norm.ppf(alpha)
    phi_z = norm.pdf(z_score)

    # VaR
    var_percentile = est_mu + z_score * est_sigma
    var_abs = abs(var_percentile)  # as distance from 0

    # ES
    es_abs = -est_mu + est_sigma * phi_z / alpha  # as distance from 0
    
    return var_abs, es_abs

In [146]:
# VaR & ES from T dist
def var_es_t(data, alpha):
    est_nu, est_mu, est_sigma = t.fit(data)
    
    t_stat = t.ppf(alpha, df=est_nu)
    phi_t = t.pdf(t_stat, df=est_nu)

    # VaR
    var_percentile = est_mu + t_stat * est_sigma
    var_abs = abs(var_percentile)

    # ES
    es_abs = -est_mu + est_sigma * ((est_nu + t_stat**2) / (est_nu - 1)) * (phi_t / alpha)
    
    return var_abs, es_abs

In [147]:
# normal, 0.05
var_abs, es_abs = var_es_normal(data1, 0.05)
print(var_abs, es_abs)

0.052719319464728166 0.06597708780710788


In [148]:
# normal, 0.01
var_abs, es_abs = var_es_normal(data1, 0.01)
print(var_abs, es_abs)

0.07434165846073715 0.08509315496336886


In [149]:
# T, 0.05
var_abs, es_abs = var_es_t(data1, 0.05)
print(var_abs, es_abs)

0.05190077607440217 0.06771381526084028


In [150]:
# T, 0.01
var_abs, es_abs = var_es_t(data1, 0.01)
print(var_abs, es_abs)

0.07731993653418916 0.09232558184803598


e. (3) Calculate the 5% and 1% VaR and ES for the data using historical simulation.

In [151]:
def var_es_sim(data, alpha):
    np.random.seed(42)
    n_sims = 100_000
    
    q_alpha = np.quantile(data, alpha)
    VaR = -q_alpha

    # ES
    es_abs = abs(data[data <= q_alpha].mean())
    
    return var_abs, es_abs

In [152]:
# 0.05
var_abs, es_abs = var_es_sim(data1, 0.05)
print(var_abs, es_abs)

0.07731993653418916 0.07107272387024315


In [153]:
# 0.01
var_abs, es_abs = var_es_sim(data1, 0.01)
print(var_abs, es_abs)

0.07731993653418916 0.09905041150582908


f. (4) Compare the results in parts (d) and (e). How does this line up with your choice in part (b) and results in
part (c)?

T distribution is a better fit and an alpha of 1% is a better threshold, as 1% VaR and ES from T distribution is closest to that from historical simulation.

#### 2. (20 pts)
Using problem2.csv: </br>
You and your team have done research into the speed at which correlations and variances change through
time. You have found that the speed of change is different for correlations versus variances. Correlations are
slower moving but variances update faster. </br>
a. (5) Given that you have decided to use an exponentially weighted correlation and variance estimator, and
will use a different lambda for each, should you choose a higher or lower lambda for the correlation
estimator? Why?

I would choose a higher lambda for the correlation estimator, because higher lambda corresponds to a slower updating.

b. (5) Given your choice in part (a), and possible $\lambda$ values of 0.94 and 0.97, calculate the exponentially
weighted correlation for the input data.

In [154]:
data2 = pd.read_csv('problem2.csv')
data2

,x1,x2,x3,x4,x5
0,-0.011904,-0.012926,-0.006511,-0.006156,0.009584
1,0.021849,0.037438,-0.004951,-0.037799,-0.002164
2,-0.004584,0.016381,0.006534,-0.007579,-0.001832
3,0.006344,0.013604,-0.008160,-0.003432,0.008191
4,-0.025164,-0.037081,-0.001121,0.007057,0.027030
...,...,...,...,...,...
995,0.016771,0.034288,0.020311,-0.014416,-0.001094
996,0.044867,0.050742,-0.019981,-0.040901,-0.010510
997,-0.020994,-0.007120,0.004023,0.003226,0.000422
998,-0.012522,-0.020195,-0.002556,-0.021653,0.017304


In [155]:
var_lambda = 0.94
corr_lambda = 0.97

ew_corr_last = exam.ew_corr(data2, corr_lambda)
ew_corr_last

,x1,x2,x3,x4,x5
x1,1.000000,0.501728,-0.276033,-0.153536,-0.287344
x2,0.501728,1.000000,0.312724,-0.321841,-0.222609
x3,-0.276033,0.312724,1.000000,-0.073139,0.418567
x4,-0.153536,-0.321841,-0.073139,1.000000,-0.406487
x5,-0.287344,-0.222609,0.418567,-0.406487,1.000000


c. (5) Given your choice in part (a), and possible $\lambda$ values of 0.94 and 0.97, calculate the exponentially
weighted variance for the input data.

In [156]:
ew_cov_biased, var_biased = exam.ew_cov_var(data2, var_lambda)
var_biased

x1    0.000353
x2    0.000555
x3    0.000187
x4    0.000420
x5    0.000126
Name: 999, dtype: float64

d. (5) Combine the results in a final covariance matrix.

In [157]:
ew_sd = np.sqrt(var_biased)
D = np.diag(ew_sd)
cov = D @ ew_corr_last @ D

cov

,0,1,2,3,4
0,0.000353,0.000222,-0.000071,-0.000059,-0.000061
1,0.000222,0.000555,0.000101,-0.000155,-0.000059
2,-0.000071,0.000101,0.000187,-0.000020,0.000064
3,-0.000059,-0.000155,-0.000020,0.000420,-0.000094
4,-0.000061,-0.000059,0.000064,-0.000094,0.000126


#### 3. (20 pts) </br>
Using problem3.csv:
You are given the input covariance matrix and need to use it for risk analysis. </br>
a. (5) Calculate the eigenvalues of the covariance matrix. What do you see?

In [158]:
data3 = pd.read_csv('problem3.csv')
data3

,x1,x2,x3,x4,x5
0,0.000338,0.000384,-0.000078,-0.000043,-0.000046
1,0.000384,0.000501,0.000014,-0.000163,0.000066
2,-0.000078,0.000014,0.000189,-0.000101,-0.000050
3,-0.000043,-0.000163,-0.000101,0.000403,-0.000029
4,-0.000046,0.000066,-0.000050,-0.000029,0.000129


In [159]:
eigvals = np.linalg.eigvalsh(data3)
eigvals

array([-4.56160014e-05,  1.23566772e-04,  1.90829637e-04,  4.26728160e-04,
        8.64915610e-04])

This matrix is non-PSD because the smallest eigenvalue is negative.

b. (5) Calculate the nearest PSD matrix using Higham's algorithm. How does this change the eigenvalues?

In [160]:
std = np.sqrt(np.diag(data3))
corr = data3 / np.outer(std, std)
higham_corr = corr_nearest(corr)
D = np.diag(std)
higham_cov = D @ higham_corr @ D

pd.DataFrame(higham_cov)

/opt/anaconda3/lib/python3.13/site-packages/statsmodels/stats/correlation_tools.py:89: IterationLimitWarning: 
Maximum iteration reached.

  warnings.warn(iteration_limit_doc, IterationLimitWarning)


,0,1,2,3,4
0,0.000338,0.000346,-0.000068,-0.000043,-0.000036
1,0.000346,0.000501,0.000002,-0.000162,0.000055
2,-0.000068,0.000002,0.000189,-0.000101,-0.000046
3,-0.000043,-0.000162,-0.000101,0.000403,-0.000029
4,-0.000036,0.000055,-0.000046,-0.000029,0.000129


The Higham covariance matrix is PSD.

c. (5) Calculate the nearest PSD matrix using the Near PSD method of Rebonato and Jackel. How does this
change the eigenvalues?

In [161]:
cov_psd, corr_psd = exam.near_psd(data3)
pd.DataFrame(cov_psd)

,0,1,2,3,4
0,0.000338,0.000340,-0.000068,-0.000041,-0.000038
1,0.000340,0.000501,0.000006,-0.000158,0.000056
2,-0.000068,0.000006,0.000189,-0.000100,-0.000046
3,-0.000041,-0.000158,-0.000100,0.000403,-0.000029
4,-0.000038,0.000056,-0.000046,-0.000029,0.000129


The nearest PSD covariance matrix is also PSD and is very close to the Higham matrix.

d. (5) Compare the results in parts (b) and (c). Which method do you prefer and why?

I prefer Higham because the matrix from nearest PSD is not necessarily the closest matrix to the original.

#### 4. (60 pts) </br>
Using the price data in problem4.csv: </br>
You hold a portfolio with a current value of 100,000. </br>
a. (5) Calculate the number of shares (fractions of shares are OK) of each of these stocks in your portfolio so
that each stock has an equal weight.

In [162]:
data4 = pd.read_csv('problem4.csv')
data4

,Date,SPY,AAPL,MSFT,GOOGL,BABA
0,2022-01-03,451.875183,178.103668,323.898376,143.904205,114.324287
1,2022-01-04,451.723785,175.843262,318.344513,143.316650,113.545532
2,2022-01-05,443.049805,171.165817,306.123901,136.741821,115.065048
3,2022-01-06,442.633514,168.308487,303.704895,136.714508,120.259880
4,2022-01-07,440.883575,168.474838,303.859802,135.989517,123.279907
...,...,...,...,...,...,...
1032,2026-02-13,681.750000,255.779999,401.320007,305.720001,155.729996
1033,2026-02-17,682.849976,263.880005,396.859985,302.019989,155.429993
1034,2026-02-18,686.289978,264.350006,399.600006,303.329987,155.770004
1035,2026-02-19,684.479980,260.579987,398.459991,302.850006,154.270004


In [163]:
V0 = 100000
V = V0 / 5
shares_SPY = V / data4.loc[1036, 'SPY']
shares_AAPL = V / data4.loc[1036, 'AAPL']
shares_MSFT = V / data4.loc[1036, 'MSFT']
shares_GOOGL = V / data4.loc[1036, 'GOOGL']
shares_BABA = V / data4.loc[1036, 'BABA']

print(shares_SPY, shares_AAPL, shares_MSFT, shares_GOOGL, shares_BABA)

29.009471900659555 75.59150735135944 50.348663109163624 63.49609277544952 129.49174745987344


b. (10) Calculate the daily returns of each stock using arithmetic returns. Show the first 5 rows and last 5 rows
of the return data.

In [164]:
returns = exam.arith_return(data4)
returns

,SPY,AAPL,MSFT,GOOGL,BABA
Date,,,,,
2022-01-04,-0.000335,-0.012692,-0.017147,-0.004083,-0.006812
2022-01-05,-0.019202,-0.026600,-0.038388,-0.045876,0.013382
2022-01-06,-0.000940,-0.016693,-0.007902,-0.000200,0.045147
2022-01-07,-0.003953,0.000988,0.000510,-0.005303,0.025113
2022-01-10,-0.001244,0.000116,0.000733,0.012060,-0.011632
...,...,...,...,...,...
2026-02-13,0.000705,-0.022733,-0.001294,-0.010615,-0.018900
2026-02-17,0.001613,0.031668,-0.011113,-0.012103,-0.001926
2026-02-18,0.005038,0.001781,0.006904,0.004337,0.002188


c. (15) Remove the mean from each series. Fit both a normal and a t-distribution to the returns of each stock.
Show the parameters of best fit for each stock.

In [165]:
demeaned = pd.DataFrame()
demeaned['SPY'] = returns['SPY'] - returns['SPY'].mean()
demeaned['AAPL'] = returns['AAPL'] - returns['AAPL'].mean()
demeaned['MSFT'] = returns['MSFT'] - returns['MSFT'].mean()
demeaned['GOOGL'] = returns['GOOGL'] - returns['GOOGL'].mean()
demeaned['BABA'] = returns['BABA'] - returns['BABA'].mean()

demeaned

,SPY,AAPL,MSFT,GOOGL,BABA
Date,,,,,
2022-01-04,-0.000806,-0.013234,-0.017491,-0.005047,-0.007629
2022-01-05,-0.019673,-0.027142,-0.038732,-0.046840,0.012565
2022-01-06,-0.001410,-0.017236,-0.008246,-0.001164,0.044330
2022-01-07,-0.004424,0.000446,0.000166,-0.006267,0.024295
2022-01-10,-0.001715,-0.000426,0.000388,0.011096,-0.012449
...,...,...,...,...,...
2026-02-13,0.000234,-0.023276,-0.001638,-0.011579,-0.019717
2026-02-17,0.001143,0.031126,-0.011458,-0.013067,-0.002744
2026-02-18,0.004567,0.001239,0.006560,0.003373,0.001370


In [166]:
def fit_demean(data):
    # normal
    est_mu = np.mean(data)
    est_sigma = np.std(data, ddof=1)

    # T
    est_nu, est_mu, est_sigma = t.fit(data)
    return est_mu, est_sigma, est_nu, est_mu, est_sigma

In [167]:
exam.model_selection_metrics(demeaned['SPY'])

(np.float64(-6357.216503669188),
 np.float64(-6347.330258823549),
 np.float64(-6357.204887018655),
 np.float64(-6534.901875776858),
 np.float64(-6520.0725085084),
 np.float64(-6534.878619962905))

In [168]:
exam.model_selection_metrics(demeaned['AAPL'])

(np.float64(-5386.839781245131),
 np.float64(-5376.953536399493),
 np.float64(-5386.828164594599),
 np.float64(-5558.550043182095),
 np.float64(-5543.720675913636),
 np.float64(-5558.526787368141))

In [169]:
exam.model_selection_metrics(demeaned['MSFT'])

(np.float64(-5479.5083118152415),
 np.float64(-5469.622066969603),
 np.float64(-5479.496695164709),
 np.float64(-5601.434808892525),
 np.float64(-5586.605441624067),
 np.float64(-5601.411553078572))

In [170]:
exam.model_selection_metrics(demeaned['GOOGL'])

(np.float64(-5122.192470565705),
 np.float64(-5112.306225720066),
 np.float64(-5122.1808539151725),
 np.float64(-5222.773812277924),
 np.float64(-5207.944445009465),
 np.float64(-5222.7505564639705))

In [171]:
exam.model_selection_metrics(demeaned['BABA'])

(np.float64(-4126.408629357199),
 np.float64(-4116.522384511561),
 np.float64(-4126.397012706667),
 np.float64(-4377.4728582159605),
 np.float64(-4362.643490947502),
 np.float64(-4377.449602402007))

In [172]:
SPY_mu, SPY_sigma, SPY_nu, SPY_mu, SPY_sigma = fit_demean(demeaned['SPY'])  # T
AAPL_mu, AAPL_sigma, AAPL_nu, AAPL_mu, AAPL_sigma = fit_demean(demeaned['AAPL'])  # T
MSFT_mu, MSFT_sigma, MSFT_nu, MSFT_mu, MSFT_sigma = fit_demean(demeaned['MSFT'])  # T
G_mu, G_sigma, G_nu, G_mu, G_sigma = fit_demean(demeaned['GOOGL'])  # T
BABA_mu, BABA_sigma, BABA_nu, BABA_mu, BABA_sigma = fit_demean(demeaned['BABA'])  # T

print(SPY_nu, SPY_mu, SPY_sigma)
print(AAPL_nu, AAPL_mu, AAPL_sigma)
print(MSFT_nu, MSFT_mu, MSFT_sigma)
print(G_nu, G_mu, G_sigma)
print(BABA_nu, BABA_mu, BABA_sigma)

3.6649407268260443 0.0002785775955092724 0.007722803958883931
3.4929117788471684 0.0001687410589990753 0.012190223147722019
4.12632236364186 0.00020620892165817165 0.012529106056809985
4.552050899028185 3.522499377090629e-05 0.015416457363624788
3.1251434547093817 -0.002160441993455156 0.020769537071147384


d. (30) Calculate the 1% VaR and ES for each stock and the portfolio as $ values. Use a Gaussian copula to
model the dependence structure between the stocks. Calculate the 1% VaR and ES using a historical
simulation as well. Present the results in a table and compare the results. Which method do you prefer and
why?

In [173]:
# Gaussian Copula
def copula(returns):
    # transform to uniforms
    U_S = t.cdf(S_return, SPY_nu, SPY_mu, SPY_sigma)
    U_A = t.cdf(S_return, AAPL_nu, AAPL_mu, AAPL_sigma)
    U_M = t.cdf(M_return, MSFT_nu, MSFT_mu, MSFT_sigma)
    U_G = t.cdf(G_return, G_nu, G_mu, G_sigma)
    U_B = t.cdf(B_return, BABA_nu, BABA_mu, BABA_sigma)
    U = np.column_stack([U_S, U_A, U_M, U_G, U_B])

    # Gaussian Copula corr
    Z = norm.ppf(np.column_stack([U_S, U_A, U_M, U_G, U_B]))
    copula_corr = np.corrcoef(Z.T)  

    # simulate joint uniforms
    np.random.seed(42)
    n_sims = 100_000
    sim = multivariate_normal.rvs(np.zeros(5), copula_corr, n_sims)
    U_sim = norm.cdf(sim)

    # transform back to T dist
    S_sim = t.ppf(U_sim[:, 1], SPY_nu, SPY_mu, SPY_sigma)
    A_sim = t.ppf(U_sim[:, 1], AAPL_nu, AAPL_mu, AAPL_sigma)
    M_sim = t.ppf(U_sim[:, 1], MSFT_nu, MSFT_mu, MSFT_sigma)
    G_sim = t.ppf(U_sim[:, 1], G_nu, G_mu, G_sigma)
    B_sim = t.ppf(U_sim[:, 1], BABA_nu, BABA_mu, BABA_sigma)
    
    R_sim = np.column_stack([S_sim, A_sim, M_sim, G_sim, B_sim])

    # equal weight: 1 / 5 = 0.2
    R_port = 0.2 * (S_sim + A_sim + M_sim + G_sim + B_sim)

    return S_sim, A_sim, M_sim, G_sim, B_sim, R_port

result = copula(demeaned)

In [174]:
# Copula - Each stock's VaR & ES
def var_es(sim, pos_value, nu, mu, sigma):
    alpha = 0.01
    var_pct = abs(np.quantile(sim, alpha))
    var = var_pct * pos_value
        
    t_stat = t.ppf(alpha, df=nu)
    phi_t = t.pdf(t_stat, df=nu)
    es_pct = -mu + sigma * ((nu + t_stat**2) / (nu - 1)) * (phi_t / alpha)
    es = es_pct * pos_value
    return var, es, var_pct, es_pct

SPY_var, SPY_es, SPY_var_pct, SPY_es_pct = var_es(result[0], V, SPY_nu, SPY_mu, SPY_sigma)
AAPL_var, AAPL_es, AAPL_var_pct, AAPL_es_pct = var_es(result[1], V, AAPL_nu, AAPL_mu, AAPL_sigma)
MSFT_var, MSFT_es, MSFT_var_pct, MSFT_es_pct = var_es(result[2], V, MSFT_nu, MSFT_mu, MSFT_sigma)
G_var, G_es, G_var_pct, G_es_pct = var_es(result[3], V, G_nu, G_mu, G_sigma)
BABA_var, BABA_es, BABA_var_pct, BABA_es_pct = var_es(result[4], V, BABA_nu, BABA_mu, BABA_sigma)
                                      
print(SPY_var, SPY_es, SPY_var_pct, SPY_es_pct)
print(AAPL_var, AAPL_es, AAPL_var_pct, AAPL_es_pct)
print(MSFT_var, MSFT_es, MSFT_var_pct, MSFT_es_pct)
print(G_var, G_es, G_var_pct, G_es_pct)
print(BABA_var, BABA_es, BABA_var_pct, BABA_es_pct)

611.8299637879055 865.3290900056085 0.030591498189395275 0.043266454500280424
1001.9093890253266 1436.7924772181912 0.05009546945126633 0.07183962386090956
930.8195732817725 1271.5202751361107 0.04654097866408862 0.06357601375680554
1093.8701149329693 1458.6453378012804 0.05469350574664846 0.07293226689006402
1898.0400675520357 2811.8460446732906 0.09490200337760178 0.14059230223366453


In [175]:
# Copula - Portfolio VaR & ES
def copula_output(returns):
    alpha = 0.01
    R_port = result[5]
    
    # total VaR & ES
    q_alpha = np.quantile(R_port, alpha)
    var_pct = -q_alpha
    es_pct = -R_port[R_port <= q_alpha].mean()
    var = V0 * var_pct
    es = V0 * es_pct
    
    return var, es, var_pct, es_pct

var, es, var_pct, es_pct = copula_output(demeaned)
print(var, es, var_pct, es_pct)

5536.46910858001 7825.155497737478 0.0553646910858001 0.07825155497737478


In [176]:
# historical simulation - Portfolio VaR & ES
def hist_sim(returns):
    alpha = 0.01

    # equal weight: 1 / 5 = 0.2
    weights = np.array([0.2, 0.2, 0.2, 0.2, 0.2])
    R_port = returns @ weights

    # total VaR & ES
    q_alpha = np.quantile(R_port, alpha)
    var_pct = -q_alpha
    es_pct = -R_port[R_port <= q_alpha].mean()
    var = V0 * var_pct
    es = V0 * es_pct
    
    return var, es, var_pct, es_pct


var, es, var_pct, es_pct = hist_sim(demeaned)
print(var, es, var_pct, es_pct)

3788.3207740640073 4621.0707987838505 0.03788320774064007 0.0462107079878385


In [181]:
# historical simulation - Each stock's VaR
def var_es_sim(returns):
    alpha = 0.01
    
    q_alpha = np.quantile(returns, alpha)
    var = -V * q_alpha

    # es_pct = -data[returns <= q_alpha].mean()
    # es = V * es_pct

    return var

SPY_var = var_es_sim(demeaned['SPY'])
AAPL_var = var_es_sim(demeaned['AAPL'])
MSFT_var = var_es_sim(demeaned['MSFT'])
G_var = var_es_sim(demeaned['GOOGL'])
BABA_var = var_es_sim(demeaned['BABA'])
                                      
print(SPY_var)
print(AAPL_var)
print(MSFT_var)
print(G_var)
print(BABA_var)

634.2622101426565
973.1540191397924
869.9846408744539
1003.9869572970764
1628.2225373689691
